## 1.13 Function Calling

In [1]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK but ignored by Ollama
)

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [8]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=ollama_client,
    instructions=instructions,
)

In [ ]:
assistant.rag("How do I run Olama locally?")

In [10]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = ollama_client.responses.create(
    model="qwen3.5:9b",
    input=messages,
)

response.output_text

'Yes, you might be able to join the course! Here\'s what to check:  \n\n1️⃣ **Enrollment Status**: Some courses are open enrollment (you can join anytime), while others have fixed start/end dates.  \n2️⃣ **Deadlines**: Look for a "Starts Soon" or "Enrolling Now" banner. If there\'s a deadline, it\'s usually listed near the enrollment page.  \n3️⃣ **Prerequisites**: Ensure you meet any requirements (e.g., prior courses, skills).  \n4️⃣ **Access Method**: Is it free? Does it require a platform subscription? Look for a "Sign Up" or "Enroll" button.  \n\n🔍 **Pro Tip**: If you\'re unsure, search the course description or contact the provider\'s support team. Let me know if you\'d like help finding specific details! 😊'

In [11]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [12]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [13]:
response = ollama_client.responses.create(
    model="qwen3.5:9b",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='rs_resp_612939', summary=[Summary(text="The user is asking about whether they can join a course. This seems like a general question about course enrollment. I should search the FAQ database to see if there's information about joining the course or enrollment requirements.", type='summary_text')], type='reasoning', content=None, encrypted_content="The user is asking about whether they can join a course. This seems like a general question about course enrollment. I should search the FAQ database to see if there's information about joining the course or enrollment requirements.", status=None),
 ResponseFunctionToolCall(arguments='{"query":"join course enrollment requirements"}', call_id='call_neq1qgix', name='search', type='function_call', id='fc_resp_612939_0', namespace=None, status='completed')]

In [14]:
import json
call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [15]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [16]:
messages.append(response.output[1])
messages.append(function_call_output)


In [17]:
response = ollama_client.responses.create(
    model="qwen3.5:9b",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='rs_resp_846026', summary=[Summary(text='The search results contain a direct answer to the user\'s question: "I just discovered the course. Can I still join?". The answer is "Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions." I should provide this information to the user.', type='summary_text')], type='reasoning', content=None, encrypted_content='The search results contain a direct answer to the user\'s question: "I just discovered the course. Can I still join?". The answer is "Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions." I should provide this information to the user.', status=None),
 ResponseOutputMessage(id='msg_839110', content=[ResponseOutputText(annotations=[], text="Yes, you can join! Just keep in mind that if you want to receive a certificate, you'll need to submit your project while the course is still acce

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(952, 121)

In [18]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


## 1.14 Agentic Loop

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [33]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [36]:
response = ollama_client.responses.create(
    model="qwen3.5:9b",
    input=messages,
    tools=[search_tool],
)
response.output

[ResponseReasoningItem(id='rs_resp_150104', summary=[Summary(text="The user is asking about joining a course they just discovered. This seems like a general question about course enrollment, but I should search the FAQ database to see if there's information about joining/enrolling in the course. Let me search for relevant information.", type='summary_text')], type='reasoning', content=None, encrypted_content="The user is asking about joining a course they just discovered. This seems like a general question about course enrollment, but I should search the FAQ database to see if there's information about joining/enrolling in the course. Let me search for relevant information.", status=None),
 ResponseFunctionToolCall(arguments='{"query":"join course enrollment"}', call_id='call_xa164xic', name='search', type='function_call', id='fc_resp_150104_0', namespace=None, status='completed')]

In [37]:
# messages.extend(response.output[1])
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        messages.append(item)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment"}


In [38]:
response = ollama_client.responses.create(
    model="qwen3.5:9b",
    input=messages,
    tools=[search_tool],
)

In [46]:
response.output

[ResponseReasoningItem(id='rs_resp_633348', summary=[Summary(text='The search results show that the user can still join the course even if they just discovered it. The FAQ indicates that yes, you can still join, but to receive a certificate, you need to submit your project while the course is still accepting submissions. I should provide this information to the user in a clear and helpful way.', type='summary_text')], type='reasoning', content=None, encrypted_content='The search results show that the user can still join the course even if they just discovered it. The FAQ indicates that yes, you can still join, but to receive a certificate, you need to submit your project while the course is still accepting submissions. I should provide this information to the user in a clear and helpful way.', status=None),
 ResponseOutputMessage(id='msg_630903', content=[ResponseOutputText(annotations=[], text='Yes, you can still join the course! However, if you want to receive a certificate, you must

In [40]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enrollment"}', call_id='call_xa164xic', name='search', type='function_call', id='fc_resp_150104_0', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_xa164xic',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just disc

In [39]:
response.output

[ResponseReasoningItem(id='rs_resp_633348', summary=[Summary(text='The search results show that the user can still join the course even if they just discovered it. The FAQ indicates that yes, you can still join, but to receive a certificate, you need to submit your project while the course is still accepting submissions. I should provide this information to the user in a clear and helpful way.', type='summary_text')], type='reasoning', content=None, encrypted_content='The search results show that the user can still join the course even if they just discovered it. The FAQ indicates that yes, you can still join, but to receive a certificate, you need to submit your project while the course is still accepting submissions. I should provide this information to the user in a clear and helpful way.', status=None),
 ResponseOutputMessage(id='msg_630903', content=[ResponseOutputText(annotations=[], text='Yes, you can still join the course! However, if you want to receive a certificate, you must

### Agentic Loop

In [ ]:
def agent_loop(instructions, question, model="qwen3.5:9b") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = ollama_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )


        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)

                # 1. Store the tool call itself in history  
                messages.append(item)

                # 2. Execute the local python search function
                call_output = make_call(item)

                # 3. Store the tool output back into history
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    return last_answer


In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

In [47]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment"}
iteration #2...
ASSISTANT:
Yes, you can still join the course even if you discovered it recently. However, please keep in mind that to receive a certificate, you must submit your project while the course is still accepting submissions. Certificates are only awarded for the live cohort, not for self-paced learning.


'Yes, you can still join the course even if you discovered it recently. However, please keep in mind that to receive a certificate, you must submit your project while the course is still accepting submissions. Certificates are only awarded for the live cohort, not for self-paced learning.'

In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")


iteration #1...
function_call: search {"query":"join course enroll"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! You can start learning and even submit homework while the registration form is open.

However, please note that if your goal is to receive a **certificate**, you will need to submit your project while the course is still accepting submissions to participate in the live cohort's peer review process. Certificates are only awarded to those who complete the course with a "live" cohort, not for self-paced participants.

Since you are just discovering the course now, check if the registration form is still open so you can secure a spot and aim for a certificate if you wish.


'Yes, you can still join the course! You can start learning and even submit homework while the registration form is open.\n\nHowever, please note that if your goal is to receive a **certificate**, you will need to submit your project while the course is still accepting submissions to participate in the live cohort\'s peer review process. Certificates are only awarded to those who complete the course with a "live" cohort, not for self-paced participants.\n\nSince you are just discovering the course now, check if the registration form is still open so you can secure a spot and aim for a certificate if you wish.'

In [50]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally"}
iteration #2...
function_call: search {"query":"Olama"}
iteration #3...
ASSISTANT:
To run **Olama** locally, follow these steps:

1.  **Install Olama**
    First, download and install the Olama CLI tool for your operating system.
    -   **Linux / macOS**:
        ```bash
        curl -L https://ollama.com/install.sh | sh
        ```
    -   **Windows**:
        Download the installer from the [official Olama GitHub releases](https://github.com/ollama/ollama/releases) or via **Scoop** / **Chocolatey**.

2.  **Start the Service**
    Once installed, start the Olama service. On most systems, it runs as a background process automatically. You can check its status or stop/start it manually:
    -   **Check Status**:
        ```bash
        ollama list
        ```
    -   **Start Service** (if needed):
        ```bash
        ollama serve
        ```

3.  **Pull and Run Models**
    -   **Pull a Model**:
        ```bash
   

'To run **Olama** locally, follow these steps:\n\n1.  **Install Olama**\n    First, download and install the Olama CLI tool for your operating system.\n    -   **Linux / macOS**:\n        ```bash\n        curl -L https://ollama.com/install.sh | sh\n        ```\n    -   **Windows**:\n        Download the installer from the [official Olama GitHub releases](https://github.com/ollama/ollama/releases) or via **Scoop** / **Chocolatey**.\n\n2.  **Start the Service**\n    Once installed, start the Olama service. On most systems, it runs as a background process automatically. You can check its status or stop/start it manually:\n    -   **Check Status**:\n        ```bash\n        ollama list\n        ```\n    -   **Start Service** (if needed):\n        ```bash\n        ollama serve\n        ```\n\n3.  **Pull and Run Models**\n    -   **Pull a Model**:\n        ```bash\n        ollama pull <model_name>\n        ```\n        (e.g., `ollama pull llama2` or `ollama pull mistral`)\n    -   **Run a Mo

In [51]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
The **Queen's Gambit** is a classic chess opening that occurs after the moves 1. d4 d5 2. c4. 

In this opening:
- White (the one with the "Queen's Gambit" name) plays c4, offering a pawn to Black
- The idea is to gain central control and potentially develop pieces advantageously
- Black can either accept the gambit by taking the pawn (dxc4) or decline it

It's one of the most popular and well-known chess openings, and it gained renewed popularity in recent years due to the popular Netflix series "Queen's Gambit," though that series is more about chess tournaments and strategy rather than specifically teaching this particular opening.


'The **Queen\'s Gambit** is a classic chess opening that occurs after the moves 1. d4 d5 2. c4. \n\nIn this opening:\n- White (the one with the "Queen\'s Gambit" name) plays c4, offering a pawn to Black\n- The idea is to gain central control and potentially develop pieces advantageously\n- Black can either accept the gambit by taking the pawn (dxc4) or decline it\n\nIt\'s one of the most popular and well-known chess openings, and it gained renewed popularity in recent years due to the popular Netflix series "Queen\'s Gambit," though that series is more about chess tournaments and strategy rather than specifically teaching this particular opening.'

In [52]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.


Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I couldn't find any information about "Queen's Gambit" in the course FAQ database.

This term usually refers to either:
1. **Queen's Gambit** - A common opening in chess where White offers the c4 pawn to challenge Black's control of the center
2. **Queen's Gambit** - A popular Netflix series about an orphaned chess prodigy from the 1960s

If you're looking for information about these topics in the context of a specific course or curriculum, you might want to check with your instructor or explore other resources. Is there something specific about chess or the show you're trying to learn more about?


'I couldn\'t find any information about "Queen\'s Gambit" in the course FAQ database.\n\nThis term usually refers to either:\n1. **Queen\'s Gambit** - A common opening in chess where White offers the c4 pawn to challenge Black\'s control of the center\n2. **Queen\'s Gambit** - A popular Netflix series about an orphaned chess prodigy from the 1960s\n\nIf you\'re looking for information about these topics in the context of a specific course or curriculum, you might want to check with your instructor or explore other resources. Is there something specific about chess or the show you\'re trying to learn more about?'

## 1.15 FrameWorks (ToyAIKit)

In [53]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [54]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [56]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [58]:
# if we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for the entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [59]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for the entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [63]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

llm_client = OpenAIClient(
    model="qwen3.5:9b",
    client=ollama_client
)

In [64]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [66]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


OS,Installation Method
macOS,Download the .pkg and install it
Windows,Download the .msi and install it
Linux,Run: curl -fsSL https://ollama.com/install.sh \| sh


d:\projects\intro-to-rag\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model '9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [67]:
result.cost

In [68]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='rs_resp_746252', summary=[Summary(text='The user is asking about how to run Olam

In [69]:
#Continuing the conversation
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)


-> Response received


-> Response received


Model,Command,Size
Llama 3,ollama run llama3,~4.7GB
Mistral,ollama run mistral,~4.1GB
Gemma,ollama run gemma:2b,~1.8GB
Phi,ollama run phi3,~2.4GB
Command R,ollama run command-r,~56GB


d:\projects\intro-to-rag\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model '9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [70]:
runner.run()

-> Response received


-> Response received


d:\projects\intro-to-rag\.venv\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model '9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


KeyboardInterrupt: 